In [ ]:
# Cell 1: prepare the data before building the model
from datasets import load_dataset
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score

# Use the official train, validation, and test splits.
dataset = load_dataset("google-research-datasets/go_emotions", "simplified")
label_names = dataset["train"].features["labels"].feature.names
assert label_names == GOEMOTIONS_LABELS

# Each sentence needs 28 targets: 1 = present, 0 = absent.
train = dataset["train"].to_pandas()
train_labels = np.zeros((len(train), len(label_names)), dtype=np.float32)

for row, labels in enumerate(dataset["train"]["labels"]):
    train_labels[row, labels] = 1

class_counts = train_labels.sum(axis=0)


In [ ]:
# Cell 2: prepare batches 
batch_size = 8
max_length = 64

# This function prepares one batch of sentences and their labels.
def prepare_batch(examples):
    texts = [example["text"] for example in examples]
    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )

    labels = torch.zeros(len(examples), len(label_names))
    for row, example in enumerate(examples):
        labels[row, example["labels"]] = 1

    return inputs, labels

train_loader = DataLoader(
    dataset["train"], batch_size=batch_size, shuffle=True, collate_fn=prepare_batch
)
val_loader = DataLoader(
    dataset["validation"], batch_size=batch_size, collate_fn=prepare_batch
)
test_loader = DataLoader(
    dataset["test"], batch_size=batch_size, collate_fn=prepare_batch
)


In [ ]:
# Cell 3: choose the training settings
from pathlib import Path
from datetime import datetime

# A new folder for each run keeps previous results safe.
results_folder = Path("training_results") / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
results_folder.mkdir(parents=True)
best_model_path = results_folder / "best_model.pt"

assert not cfg.held_out_classes, "This loop trains all 28 labels."
assert torch.all(model.fusion.seen_mask == 1)
model = model.to(device)

# Use the loss already defined in  classifier notebook.
seen_mask = model.fusion.seen_mask.detach().cpu()
loss_fn = build_loss_fn(cfg, class_counts, seen_mask).to(device)

# The optimizer changes the model weights after each batch.
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
epochs = 7
best_f1 = -1.0
history = []


In [ ]:
# Cell 4: train, validate, and save the best model
for epoch in range(epochs):
    # TRAINING
    model.train()
    total_loss = 0.0
    total_examples = 0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()                         # Clear old gradients.
        outputs = model(inputs["input_ids"], inputs["attention_mask"])
        loss = loss_fn(outputs["z_final"], labels)      # Measure prediction errors.
        loss.backward()                               # Calculate gradients.
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()                              # Update the weights.

        total_loss += loss.item() * len(labels)
        total_examples += len(labels)

    train_loss = total_loss / total_examples

    # VALIDATION: check performance without updating weights.
    model.eval()
    val_predictions = []
    val_labels = []

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            outputs = model(inputs["input_ids"], inputs["attention_mask"])
            probabilities = torch.sigmoid(outputs["z_final"])
            predictions = (probabilities >= 0.5).int()

            val_predictions.extend(predictions.cpu().tolist())
            val_labels.extend(labels.tolist())

    val_f1 = f1_score(val_labels, val_predictions, average="macro", zero_division=0)
    print(f"Epoch {epoch + 1}/{epochs} | Loss: {train_loss:.4f} | Val Macro-F1: {val_f1:.4f}")
    history.append({"epoch": epoch + 1, "train_loss": train_loss, "val_macro_f1": val_f1})
    pd.DataFrame(history).to_csv(results_folder / "history.csv", index=False)

    # Keep the weights from the epoch with the highest validation F1.
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), best_model_path)
        print("Saved the best model so far.")


In [ ]:
# Cell 5: choose a threshold on validation, then evaluate on test
import json

# Use the best epoch, which may not be the last epoch.
model.load_state_dict(torch.load(best_model_path, map_location=device, weights_only=True))
model.eval()

# This helper collects predictions without changing model weights.
def get_probabilities(loader):
    all_probabilities = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            outputs = model(inputs["input_ids"], inputs["attention_mask"])
            probabilities = torch.sigmoid(outputs["z_final"])
            all_probabilities.extend(probabilities.cpu().tolist())
            all_labels.extend(labels.tolist())

    return np.array(all_probabilities), np.array(all_labels)

val_probabilities, val_labels = get_probabilities(val_loader)
best_threshold = 0.5
best_threshold_f1 = -1.0

# Pick the threshold using validation only.
for threshold in np.linspace(0.05, 0.95, 19):
    predictions = (val_probabilities >= threshold).astype(int)
    score = f1_score(val_labels, predictions, average="macro", zero_division=0)
    if score > best_threshold_f1:
        best_threshold_f1 = score
        best_threshold = float(threshold)

# Keep that threshold fixed for the test set.
test_probabilities, test_labels = get_probabilities(test_loader)
test_predictions = (test_probabilities >= best_threshold).astype(int)
macro_f1 = f1_score(test_labels, test_predictions, average="macro", zero_division=0)
micro_f1 = f1_score(test_labels, test_predictions, average="micro", zero_division=0)

print(f"Threshold: {best_threshold:.2f}")
print(f"Test Macro-F1: {macro_f1:.4f}")
print(f"Test Micro-F1: {micro_f1:.4f}")

results = {
    "threshold": best_threshold,
    "test_macro_f1": float(macro_f1),
    "test_micro_f1": float(micro_f1),
    "label_names": label_names,
    "max_length": max_length,
}
with open(results_folder / "results.json", "w") as file:
    json.dump(results, file, indent=2)
tokenizer.save_pretrained(results_folder / "tokenizer")
print("Saved results in:", results_folder)
